In [15]:
import ee
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon, Point
import json
import requests
from datetime import datetime
import os

# 1. INICIALIZACIÓN
PROJECT_ID = 'gen-lang-client-0247558984' 

try:
    ee.Initialize(
        project=PROJECT_ID,
        opt_url='https://earthengine-highvolume.googleapis.com'
    )
    print(f"✅ Conectado a Google Earth Engine (Proyecto: {PROJECT_ID})")
except Exception as e:
    print("⚠️ Error: Necesitas configurar tu proyecto GEE")
    print("Ve a: https://code.earthengine.google.com/register")
    raise


✅ Conectado a Google Earth Engine (Proyecto: gen-lang-client-0247558984)


In [17]:
# 2. CONFIGURACIÓN
# Región delimitada en Google Earth (Arequipa, Perú)


coords = [
    [-71.3385389,  -16.43038162],
    [-71.33685562, -16.43528126],
    [-71.32475405, -16.43134153],
    [-71.3256297,  -16.42287724],
    [-71.32898083, -16.42349868],
    [-71.33447786, -16.42826592],
    [-71.3385389,  -16.43038162]  # cierre del polígono
]


roi = ee.Geometry.Polygon([coords], None, False)

fecha_inicio = '2024-01-01'
fecha_fin = '2026-01-14'

# Carpeta de salida
OUTPUT_DIR = 'datos_reforestacion_gis'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# TODAS LAS BANDAS ESPECTRALES
bandas_espectrales = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

# 3. FUNCIONES
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    
    scl = image.select('SCL')
    scl_mask = scl.gte(4).And(scl.lte(6))
    combined_mask = mask.And(scl_mask)
    
    return image.updateMask(combined_mask).divide(10000).select(bandas_espectrales) \
                .copyProperties(image, ['system:time_start', 'CLOUDY_PIXEL_PERCENTAGE'])

def add_all_indices(image):
    # ÍNDICES DE VEGETACIÓN
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    evi = image.expression(
        '2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('EVI')
    
    savi = image.expression(
        '((NIR - RED) / (NIR + RED + 0.428)) * (1.428)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4')
        }).rename('SAVI')
    
    ndre = image.normalizedDifference(['B8', 'B5']).rename('NDRE')
    gndvi = image.normalizedDifference(['B8', 'B3']).rename('GNDVI')
    
    cire = image.expression(
        '(NIR / RE1) - 1', {
            'NIR': image.select('B8'),
            'RE1': image.select('B5')
        }).rename('CIre')
    
    # ÍNDICES DE AGUA/HUMEDAD
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    ndmi = image.normalizedDifference(['B8', 'B11']).rename('NDMI')
    
    msi = image.expression(
        'SWIR1 / NIR', {
            'SWIR1': image.select('B11'),
            'NIR': image.select('B8')
        }).rename('MSI')
    
    # ÍNDICES DE SUELO
    nbr = image.normalizedDifference(['B8', 'B12']).rename('NBR')
    
    bsi = image.expression(
        '((SWIR1 + RED) - (NIR + BLUE)) / ((SWIR1 + RED) + (NIR + BLUE))', {
            'SWIR1': image.select('B11'),
            'RED': image.select('B4'),
            'NIR': image.select('B8'),
            'BLUE': image.select('B2')
        }).rename('BSI')
    
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    
    # ÍNDICES ADICIONALES
    lai = image.expression(
        '3.618 * EVI - 0.118', {
            'EVI': evi
        }).rename('LAI')
    
    arvi = image.expression(
        '(NIR - (2 * RED) + BLUE) / (NIR + (2 * RED) + BLUE)', {
            'NIR': image.select('B8'),
            'RED': image.select('B4'),
            'BLUE': image.select('B2')
        }).rename('ARVI')
    
    sipi = image.expression(
        '(NIR - BLUE) / (NIR - RED)', {
            'NIR': image.select('B8'),
            'BLUE': image.select('B2'),
            'RED': image.select('B4')
        }).rename('SIPI')
    
    psri = image.expression(
        '(RED - GREEN) / RE1', {
            'RED': image.select('B4'),
            'GREEN': image.select('B3'),
            'RE1': image.select('B6')
        }).rename('PSRI')
    
    return image.addBands([
        ndvi, evi, savi, ndre, gndvi, cire,
        ndwi, ndmi, msi,
        nbr, bsi, ndbi,
        lai, arvi, sipi, psri
    ])


GEOJSON CON FECHA

In [ ]:
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate(fecha_inicio, fecha_fin)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    .map(mask_s2_clouds)
    .map(add_all_indices)
)



indices = [
    'NDVI','EVI','SAVI','NDRE','GNDVI','CIre',
    'NDWI','NDMI','MSI',
    'NBR','BSI','NDBI',
    'LAI','ARVI','SIPI','PSRI'
]

def image_to_feature(image):
    stats = image.select(indices).reduceRegion(
        reducer = ee.Reducer.mean(),
        geometry = roi,
        scale = 10,
        maxPixels = 1e13
    )
    
    fecha = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')

    return ee.Feature(roi, stats).set({
        'date': fecha,
        'cloud_pct': image.get('CLOUDY_PIXEL_PERCENTAGE')
    })


fc = ee.FeatureCollection(collection.map(image_to_feature))



url = fc.getDownloadURL(filetype='geojson', filename='indices_temporales')


print("Descargando GeoJSON...")
r = requests.get(url)

geojson_path = os.path.join(OUTPUT_DIR, 'indices_temporales.geojson')
with open(geojson_path, 'wb') as f:
    f.write(r.content)

print("✅ GeoJSON creado:", geojson_path)


Descargando GeoJSON...
✅ GeoJSON creado: datos_reforestacion_gis\indices_temporales.geojson


In [9]:

# 4. PROCESAMIENTO
print("\n🌍 Cargando imágenes Sentinel-2...")
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(roi)
              .filterDate(fecha_inicio, fecha_fin)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
              .map(mask_s2_clouds)
              .map(add_all_indices))

n_images = collection.size().getInfo()
print(f"📸 Imágenes encontradas: {n_images}")

# 5. EXTRACCIÓN DE DATOS TABULARES
def reduce_region_function(image):
    stats_mean = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    stats_std = image.reduceRegion(
        reducer=ee.Reducer.stdDev(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    stats_minmax = image.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=10,
        maxPixels=1e9,
        bestEffort=True
    )
    
    all_stats = stats_mean.combine(stats_std).combine(stats_minmax)
    
    return ee.Feature(None, all_stats).set({
        'date': image.date().format('YYYY-MM-dd'),
        'timestamp': image.date().millis(),
        'cloud_percentage': image.get('CLOUDY_PIXEL_PERCENTAGE')
    })

print("\n📊 Procesando estadísticas temporales...")
timeseries_data = collection.map(reduce_region_function)
timeseries_data = timeseries_data.filter(ee.Filter.notNull(['NDVI']))

data_json = timeseries_data.getInfo()

if not data_json['features']:
    print("⚠️ No se encontraron datos válidos")
else:
    data_list = [item['properties'] for item in data_json['features']]
    df = pd.DataFrame(data_list)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    
    print(f"✅ {len(df)} observaciones descargadas")
    



🌍 Cargando imágenes Sentinel-2...
📸 Imágenes encontradas: 298

📊 Procesando estadísticas temporales...
✅ 275 observaciones descargadas


In [10]:
    # ============================================
    # EXPORTACIÓN EN MÚLTIPLES FORMATOS GIS
    # ============================================
    
    # 1. CSV (Tabla simple)
    print("\n💾 Exportando formatos...")
    csv_file = os.path.join(OUTPUT_DIR, 'datos_temporales.csv')
    df.to_csv(csv_file, index=False)
    print(f"   ✓ CSV: {csv_file}")
    
    # 2. GEOJSON (Polígono del área + datos temporales)
    # Crear un GeoDataFrame con el polígono del ROI
    roi_polygon = Polygon(coords[:-1])  # Remover último punto duplicado
    
    # GeoJSON 1: Polígono del área de estudio
    gdf_roi = gpd.GeoDataFrame(
        {'nombre': ['Área de Reforestación Arequipa'],
         'area_km2': [roi_polygon.area * 111 * 111],  # Conversión aproximada
         'n_observaciones': [len(df)],
         'fecha_inicio': [fecha_inicio],
         'fecha_fin': [fecha_fin]},
        geometry=[roi_polygon],
        crs='EPSG:4326'
    )
    
    geojson_roi = os.path.join(OUTPUT_DIR, 'area_estudio.geojson')
    gdf_roi.to_file(geojson_roi, driver='GeoJSON')
    print(f"   ✓ GeoJSON (Área): {geojson_roi}")
    
    # GeoJSON 2: Serie temporal como puntos (centroide por fecha)
    centroid = roi_polygon.centroid
    
    gdf_temporal = gpd.GeoDataFrame(
        df,
        geometry=[Point(centroid.x, centroid.y)] * len(df),
        crs='EPSG:4326'
    )
    
    geojson_temporal = os.path.join(OUTPUT_DIR, 'serie_temporal_puntos.geojson')
    gdf_temporal.to_file(geojson_temporal, driver='GeoJSON')
    print(f"   ✓ GeoJSON (Serie temporal): {geojson_temporal}")
    
    # 3. SHAPEFILE (para ArcGIS clásico)
    shp_roi = os.path.join(OUTPUT_DIR, 'area_estudio.shp')
    gdf_roi.to_file(shp_roi)
    print(f"   ✓ Shapefile (Área): {shp_roi}")
    
    shp_temporal = os.path.join(OUTPUT_DIR, 'serie_temporal.shp')
    # Shapefile tiene límite de 10 caracteres en nombres de columna
    gdf_temporal_shp = gdf_temporal.copy()
    gdf_temporal_shp.columns = [col[:10] for col in gdf_temporal_shp.columns]
    gdf_temporal_shp.to_file(shp_temporal)
    print(f"   ✓ Shapefile (Serie temporal): {shp_temporal}")
    
    # 4. GEOPACKAGE (formato moderno, recomendado)
    gpkg_file = os.path.join(OUTPUT_DIR, 'datos_reforestacion.gpkg')
    gdf_roi.to_file(gpkg_file, layer='area_estudio', driver='GPKG')
    gdf_temporal.to_file(gpkg_file, layer='serie_temporal', driver='GPKG')
    print(f"   ✓ GeoPackage: {gpkg_file}")
    
    # 5. KML (para Google Earth)
    kml_file = os.path.join(OUTPUT_DIR, 'area_estudio.kml')
    gdf_roi.to_file(kml_file, driver='KML')
    print(f"   ✓ KML (Google Earth): {kml_file}")



💾 Exportando formatos...
   ✓ CSV: datos_reforestacion_gis\datos_temporales.csv
   ✓ GeoJSON (Área): datos_reforestacion_gis\area_estudio.geojson
   ✓ GeoJSON (Serie temporal): datos_reforestacion_gis\serie_temporal_puntos.geojson
   ✓ Shapefile (Área): datos_reforestacion_gis\area_estudio.shp
   ✓ Shapefile (Serie temporal): datos_reforestacion_gis\serie_temporal.shp

C:\Users\Borix\AppData\Local\Temp\ipykernel_36252\1769482851.py:45: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_roi.to_file(shp_roi)
c:\Users\Borix\Desktop\Proyectos\queñaris\myvenv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'n_observaciones' to 'n_observac'
  ogr_write(
c:\Users\Borix\Desktop\Proyectos\queñaris\myvenv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'fecha_inicio' to 'fecha_inic'
  ogr_write(
c:\Users\Borix\Desktop\Proyectos\queñaris\myvenv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Field date created as String field, though DateTime requested.
  ogr_write(



   ✓ GeoPackage: datos_reforestacion_gis\datos_reforestacion.gpkg
   ✓ KML (Google Earth): datos_reforestacion_gis\area_estudio.kml


In [11]:
    
    # ============================================
    # EXPORTAR IMÁGENES RASTER (GeoTIFF)
    # ============================================
    print("\n🗺️ Exportando rasters (GeoTIFF)...")
    print("   Esto puede tardar unos minutos...")
    
    # Crear imagen compuesta (mediana) de toda la serie temporal
    composite = collection.median()
    
    # Lista de índices a exportar como GeoTIFF
    indices_to_export = ['NDVI', 'EVI', 'NDWI', 'NDMI', 'NBR', 'BSI', 'LAI']
    
    for idx_name in indices_to_export:
        try:
            # Obtener la URL de descarga
            url = composite.select(idx_name).getDownloadURL({
                'scale': 10,
                'region': roi,
                'format': 'GEO_TIFF',
                'crs': 'EPSG:4326'
            })
            
            # Descargar el archivo
            response = requests.get(url)
            tiff_file = os.path.join(OUTPUT_DIR, f'{idx_name}_compuesto.tif')
            
            with open(tiff_file, 'wb') as f:
                f.write(response.content)
            
            print(f"   ✓ GeoTIFF: {idx_name}_compuesto.tif")
        except Exception as e:
            print(f"   ✗ Error en {idx_name}: {str(e)[:50]}...")
    
    # RGB True Color (para visualización)
    try:
        rgb_url = composite.select(['B4', 'B3', 'B2']).getDownloadURL({
            'scale': 10,
            'region': roi,
            'format': 'GEO_TIFF',
            'crs': 'EPSG:4326'
        })
        
        response = requests.get(rgb_url)
        rgb_file = os.path.join(OUTPUT_DIR, 'RGB_true_color.tif')
        
        with open(rgb_file, 'wb') as f:
            f.write(response.content)
        
        print(f"   ✓ GeoTIFF RGB: RGB_true_color.tif")
    except Exception as e:
        print(f"   ✗ Error en RGB: {str(e)[:50]}...")
    
    # ============================================
    # METADATA JSON
    # ============================================
    metadata = {
        'proyecto': 'Reforestación Inteligente - Arequipa',
        'fecha_generacion': datetime.now().isoformat(),
        'periodo_analisis': {
            'inicio': fecha_inicio,
            'fin': fecha_fin
        },
        'area_estudio': {
            'coordenadas': coords,
            'area_aproximada_km2': round(roi_polygon.area * 111 * 111, 2)
        },
        'datos': {
            'n_observaciones': len(df),
            'bandas_espectrales': bandas_espectrales,
            'indices_calculados': [
                'NDVI', 'EVI', 'SAVI', 'NDRE', 'GNDVI', 'CIre',
                'NDWI', 'NDMI', 'MSI', 'NBR', 'BSI', 'NDBI',
                'LAI', 'ARVI', 'SIPI', 'PSRI'
            ]
        },
        'formatos_exportados': {
            'vectoriales': ['GeoJSON', 'Shapefile', 'GeoPackage', 'KML'],
            'raster': ['GeoTIFF'],
            'tabular': ['CSV']
        },
        'sistemas_gis_compatibles': [
            'QGIS (todos los formatos)',
            'ArcGIS (Shapefile, GeoPackage, GeoTIFF)',
            'Google Earth (KML)',
            'Python/R (GeoJSON, CSV, GeoTIFF)'
        ]
    }
    
    metadata_file = os.path.join(OUTPUT_DIR, 'metadata.json')
    with open(metadata_file, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    print(f"   ✓ Metadata: {metadata_file}")
    



🗺️ Exportando rasters (GeoTIFF)...
   Esto puede tardar unos minutos...
   ✓ GeoTIFF: NDVI_compuesto.tif
   ✓ GeoTIFF: EVI_compuesto.tif
   ✓ GeoTIFF: NDWI_compuesto.tif
   ✓ GeoTIFF: NDMI_compuesto.tif
   ✓ GeoTIFF: NBR_compuesto.tif
   ✓ GeoTIFF: BSI_compuesto.tif
   ✓ GeoTIFF: LAI_compuesto.tif
   ✓ GeoTIFF RGB: RGB_true_color.tif
   ✓ Metadata: datos_reforestacion_gis\metadata.json


In [12]:
    # ============================================
    # RESUMEN FINAL
    # ============================================
    print("\n" + "="*60)
    print("✅ EXPORTACIÓN COMPLETADA")
    print("="*60)
    print(f"\n📁 Carpeta de salida: {OUTPUT_DIR}/")
    print("\n📦 Archivos generados:")
    print("   VECTORIALES:")
    print("   • area_estudio.geojson - Polígono del área (QGIS/ArcGIS)")
    print("   • serie_temporal_puntos.geojson - Datos temporales (QGIS/ArcGIS)")
    print("   • area_estudio.shp - Shapefile (ArcGIS clásico)")
    print("   • serie_temporal.shp - Shapefile temporal (ArcGIS clásico)")
    print("   • datos_reforestacion.gpkg - GeoPackage multi-capa (RECOMENDADO)")
    print("   • area_estudio.kml - Google Earth")
    print("\n   RASTER (GeoTIFF):")
    print("   • NDVI_compuesto.tif - Índice de vegetación")
    print("   • EVI_compuesto.tif - Vegetación mejorado")
    print("   • NDWI_compuesto.tif - Índice de agua")
    print("   • NDMI_compuesto.tif - Humedad")
    print("   • NBR_compuesto.tif - Áreas quemadas")
    print("   • BSI_compuesto.tif - Suelo desnudo")
    print("   • LAI_compuesto.tif - Área foliar")
    print("   • RGB_true_color.tif - Imagen color real")
    print("\n   TABULARES:")
    print("   • datos_temporales.csv - Serie temporal completa")
    print("   • metadata.json - Información del proyecto")
    
    print("\n🎯 CÓMO USAR EN QGIS/ArcGIS:")
    print("   1. Abre QGIS o ArcGIS")
    print("   2. Arrastra el archivo .gpkg o .geojson")
    print("   3. Para rasters, arrastra los archivos .tif")
    print("   4. Configura simbología según tus necesidades")
    
    print("\n📊 Estadísticas básicas:")
    print(f"   • NDVI promedio: {df['NDVI'].mean():.3f}")
    print(f"   • Humedad (NDMI): {df['NDMI'].mean():.3f}")
    print(f"   • Suelo desnudo (BSI): {df['BSI'].mean():.3f}")
    print(f"   • Cobertura de nubes: {df['cloud_percentage'].mean():.1f}%")


✅ EXPORTACIÓN COMPLETADA

📁 Carpeta de salida: datos_reforestacion_gis/

📦 Archivos generados:
   VECTORIALES:
   • area_estudio.geojson - Polígono del área (QGIS/ArcGIS)
   • serie_temporal_puntos.geojson - Datos temporales (QGIS/ArcGIS)
   • area_estudio.shp - Shapefile (ArcGIS clásico)
   • serie_temporal.shp - Shapefile temporal (ArcGIS clásico)
   • datos_reforestacion.gpkg - GeoPackage multi-capa (RECOMENDADO)
   • area_estudio.kml - Google Earth

   RASTER (GeoTIFF):
   • NDVI_compuesto.tif - Índice de vegetación
   • EVI_compuesto.tif - Vegetación mejorado
   • NDWI_compuesto.tif - Índice de agua
   • NDMI_compuesto.tif - Humedad
   • NBR_compuesto.tif - Áreas quemadas
   • BSI_compuesto.tif - Suelo desnudo
   • LAI_compuesto.tif - Área foliar
   • RGB_true_color.tif - Imagen color real

   TABULARES:
   • datos_temporales.csv - Serie temporal completa
   • metadata.json - Información del proyecto

🎯 CÓMO USAR EN QGIS/ArcGIS:
   1. Abre QGIS o ArcGIS
   2. Arrastra el archivo 